# 02. 실습: Joint Pretraining-RL Scaling Law

목표: 논문의 scaling law 형태를 작은 함수로 구현하고, 총 compute 예산에서 사전학습과 RL의 분배가 어떻게 달라지는지 봅니다.

실행 방법: 모든 셀을 순서대로 실행합니다. 외부 패키지는 필요하지 않습니다.

## 1. Toy scaling law 정의

아래 수식은 논문의 정확한 재현이 아니라 학습용 단순화입니다. 구조는 같습니다. 사전학습 loss가 낮을수록 RL 이후 성능 수준이 높고, 사전학습 토큰이 많을수록 RL compute에 대한 개선 기울기가 커집니다.

In [ ]:
import math


def pretraining_loss(parameters, tokens):
    # Chinchilla류 법칙의 형태만 빌린 toy 함수입니다.
    n = parameters / 1e6
    t = tokens / 1e9
    return 0.42 + 0.8 / math.sqrt(n) + 0.35 / math.sqrt(t)


def f_from_loss(loss):
    # loss가 낮아질수록 post-RL 기준 성능이 올라가도록 둡니다.
    return max(0.0, min(0.95, 1.15 - loss))


def g_from_scale(parameters, tokens):
    # 논문처럼 토큰 수의 영향이 모델 크기보다 더 크게 들어가도록 설정합니다.
    return -0.18 + 0.018 * math.log10(tokens) + 0.008 * math.log10(parameters)


def post_rl_reward(parameters, tokens, rl_flops, reference_flops=1e18):
    loss = pretraining_loss(parameters, tokens)
    base = f_from_loss(loss)
    slope = g_from_scale(parameters, tokens)
    reward = base + slope * (math.log10(max(rl_flops, 1.0)) - math.log10(reference_flops))
    return max(0.0, min(1.0, reward))


for tokens in [2e8, 1e9, 5e9, 2e10]:
    reward = post_rl_reward(50e6, tokens, 1e18)
    print(f"N=50M T={tokens/1e9:5.1f}B -> loss={pretraining_loss(50e6, tokens):.3f}, reward={reward:.3f}")

## 2. 고정 총 compute에서 frontier 찾기

사전학습 FLOPs는 `6NT`로 계산합니다. 총 예산에서 사전학습 compute를 뺀 나머지를 RL compute로 보고 가장 높은 reward 조합을 찾습니다.

In [ ]:
def pretrain_flops(parameters, tokens):
    return 6 * parameters * tokens


def search_frontier(parameters, total_flops, token_candidates):
    rows = []
    for tokens in token_candidates:
        c_pt = pretrain_flops(parameters, tokens)
        if c_pt >= total_flops:
            continue
        c_rl = total_flops - c_pt
        reward = post_rl_reward(parameters, tokens, c_rl)
        rows.append(
            {
                "tokens": tokens,
                "pretrain_share": c_pt / total_flops,
                "rl_share": c_rl / total_flops,
                "reward": reward,
            }
        )
    return max(rows, key=lambda row: row["reward"]), rows


token_candidates = [2e8, 5e8, 1e9, 2e9, 5e9, 1e10, 2e10, 5e10]
budgets = [2e17, 5e17, 1e18, 3e18, 1e19]

print("budget_FLOPs | best_tokens_B | RL_share | predicted_reward")
print("--- | --- | --- | ---")
for budget in budgets:
    best, _rows = search_frontier(50e6, budget, token_candidates)
    print(f"{budget:.1e} | {best['tokens']/1e9:5.1f} | {best['rl_share']:.2f} | {best['reward']:.3f}")

## 3. 해석하기

toy 함수의 세부 숫자는 임의지만, tradeoff는 논문과 같은 구조입니다. 너무 일찍 RL을 시작하면 사전학습 prior가 약하고, 너무 늦게 시작하면 RL compute가 부족합니다. 좋은 recipe는 총 예산에 따라 둘 사이의 균형을 찾습니다.

In [ ]:
budget = 3e18
best, rows = search_frontier(50e6, budget, token_candidates)

for row in rows:
    marker = "<-- best" if row == best else ""
    print(
        f"T={row['tokens']/1e9:5.1f}B | pretrain={row['pretrain_share']:.2f} | "
        f"RL={row['rl_share']:.2f} | reward={row['reward']:.3f} {marker}"
    )